# Cell 1 — Install/check packages

In [1]:
import sys, subprocess

def ensure(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", pkg])

try:
    import segmentation_models_pytorch as smp
except Exception:
    ensure("segmentation-models-pytorch")
    ensure("timm")

try:
    import matplotlib.pyplot as plt
except Exception:
    ensure("matplotlib")
    import matplotlib.pyplot as plt

try:
    from sklearn.metrics import auc
except Exception:
    ensure("scikit-learn")
    from sklearn.metrics import auc

import torch
import segmentation_models_pytorch as smp

print("torch:", torch.__version__)
print("smp:", smp.__version__)
print("cuda available:", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 71.4 MB/s eta 0:00:00
torch: 2.10.0+cu128
smp: 0.5.0
cuda available: True


# Cell 2 — Imports

In [2]:
import os, glob, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader

# Cell 3 — Config

In [3]:
SEED = 7
IMG_SIZE = 448              # fair comparison size across models
NUM_WORKERS = 2
THRESHOLDS = np.round(np.arange(0.10, 0.91, 0.10), 2)   # 0.10 ... 0.90

OUT_DIR = Path("/kaggle/working/output/phase8")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(SEED)

print("Thresholds:", THRESHOLDS.tolist())
print("Output folder:", OUT_DIR)

Thresholds: [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
Output folder: /kaggle/working/output/phase8


# Cell 4 — Device selection

In [4]:
def choose_device():
    if not torch.cuda.is_available():
        print("CUDA not available -> using CPU")
        return torch.device("cpu")

    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    print("GPU:", name, "| capability:", cap)

    # Your current Kaggle sessions can land on P100 (sm_60), which breaks with the current torch build
    if cap[0] < 7:
        print("⚠️ GPU capability < 7.0 -> using CPU to avoid CUDA crash")
        return torch.device("cpu")

    return torch.device("cuda")

device = choose_device()
print("Using device:", device)

GPU: Tesla P100-PCIE-16GB | capability: (6, 0)
⚠️ GPU capability < 7.0 -> using CPU to avoid CUDA crash
Using device: cpu


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


# Cell 5 — Auto-find inputs

In [5]:
# splits
VAL_CSV = glob.glob("/kaggle/input/**/val_split.csv", recursive=True)[0]
TRAIN_CSV = glob.glob("/kaggle/input/**/train_split.csv", recursive=True)[0]

val_df = pd.read_csv(VAL_CSV)
print("VAL_CSV:", VAL_CSV)
print("TRAIN_CSV:", TRAIN_CSV)
print("val rows:", len(val_df))
print("val columns:", val_df.columns.tolist())

# checkpoints
def pick_first(patterns):
    found = []
    for pat in patterns:
        found.extend(glob.glob(pat, recursive=True))
    found = sorted(set(found))
    return found[0] if found else None

MODEL1_CKPT = pick_first([
    "/kaggle/input/**/*model-1-checkpoints*/*best*.pt",
    "/kaggle/input/**/*unet_efficientnetb3_best*.pt",
])

MODEL2_CKPT = pick_first([
    "/kaggle/input/**/*model-2-checkpoints*/*best*.pt",
    "/kaggle/input/**/*deeplabv3plus*best*.pt",
])

MODEL3_CKPT = pick_first([
    "/kaggle/input/**/*model-3-checkpoints*/*best*.pt",
    "/kaggle/input/**/*unetplusplus*best*.pt",
])

print("\nMODEL1_CKPT:", MODEL1_CKPT)
print("MODEL2_CKPT:", MODEL2_CKPT)
print("MODEL3_CKPT:", MODEL3_CKPT)

assert MODEL1_CKPT is not None, "Model-1 ckpt not found"
assert MODEL2_CKPT is not None, "Model-2 ckpt not found"
assert MODEL3_CKPT is not None, "Model-3 ckpt not found"

VAL_CSV: /kaggle/input/datasets/arman00p/phase-3-csv-files/val_split.csv
TRAIN_CSV: /kaggle/input/datasets/arman00p/phase-3-csv-files/train_split.csv
val rows: 1695
val columns: ['stem', 'image_path', 'mask_path', 'source', 'empty_mask', 'has_crack']

MODEL1_CKPT: /kaggle/input/datasets/arman00p/model-1-checkpoints-unet-efficientnet-b3/output/SAVED_MODELS/unet_efficientnetb3_best_ep029_20260219_205815.pt
MODEL2_CKPT: /kaggle/input/datasets/arman00p/model-2-checkpointsdeeplabv3plus-efficientnet-b3/deeplabv3plus_efficientnetb3_pt files/deeplabv3plus_efficientnetb3_best_20260222_102303.pt
MODEL3_CKPT: /kaggle/input/datasets/arman00p/model-3-checkpoints-datasetunet-eff-net-b3/unetplusplus_efficientnetb3_best.pt


# Cell 6 — Helper functions

In [6]:
def extract_state_dict_any(ckpt_obj):
    if isinstance(ckpt_obj, dict) and len(ckpt_obj) > 0:
        if all(isinstance(v, torch.Tensor) for v in ckpt_obj.values()):
            return ckpt_obj

    if isinstance(ckpt_obj, dict):
        for key in ["state_dict", "model_state_dict", "model", "net", "weights"]:
            if key in ckpt_obj and isinstance(ckpt_obj[key], dict):
                sd = ckpt_obj[key]
                if len(sd) > 0 and all(isinstance(v, torch.Tensor) for v in sd.values()):
                    return sd

    best = None

    def walk(obj):
        nonlocal best
        if isinstance(obj, dict):
            if len(obj) > 0 and all(isinstance(v, torch.Tensor) for v in obj.values()):
                if best is None or len(obj) > len(best):
                    best = obj
            for v in obj.values():
                walk(v)
        elif isinstance(obj, (list, tuple)):
            for v in obj:
                walk(v)

    walk(ckpt_obj)
    if best is not None:
        return best

    raise ValueError("Could not locate state_dict inside checkpoint")

def strip_prefixes(sd):
    cleaned = {}
    for k, v in sd.items():
        nk = k
        for pref in ("module.", "model.", "net."):
            if nk.startswith(pref):
                nk = nk[len(pref):]
        cleaned[nk] = v
    return cleaned

def iou_empty_aware(gt, pr, eps=1e-7):
    gt = gt.astype(bool)
    pr = pr.astype(bool)
    inter = np.logical_and(gt, pr).sum()
    union = np.logical_or(gt, pr).sum()
    if union == 0:
        return 1.0
    return float(inter / (union + eps))

def dice_empty_aware(gt, pr, eps=1e-7):
    gt = gt.astype(bool)
    pr = pr.astype(bool)
    inter = np.logical_and(gt, pr).sum()
    denom = gt.sum() + pr.sum()
    if denom == 0:
        return 1.0
    return float((2 * inter) / (denom + eps))

# Cell 7 — Dataset

In [7]:
class CrackValDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_size: int, preprocess_fn):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.preprocess_fn = preprocess_fn

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = cv2.imread(str(row["image_path"]), cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(row["image_path"])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA).astype(np.float32)

        m = cv2.imread(str(row["mask_path"]), cv2.IMREAD_GRAYSCALE)
        if m is None:
            raise FileNotFoundError(row["mask_path"])
        m = cv2.resize(m, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
        m = (m > 127).astype(np.uint8)

        img = self.preprocess_fn(img)

        img_t = torch.from_numpy(img).permute(2, 0, 1).float()
        m_t = torch.from_numpy(m).unsqueeze(0).float()
        return img_t, m_t

# Cell 8 — Robust model loader

In [8]:
def load_smp_model(arch_name, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    sd = strip_prefixes(extract_state_dict_any(ckpt))

    encoder_candidates = ["efficientnet-b3", "timm-efficientnet-b3"]
    best = None

    for enc in encoder_candidates:
        if arch_name == "Unet":
            m = smp.Unet(encoder_name=enc, encoder_weights=None, in_channels=3, classes=1, activation=None)
        elif arch_name == "DeepLabV3Plus":
            m = smp.DeepLabV3Plus(encoder_name=enc, encoder_weights=None, in_channels=3, classes=1, activation=None)
        elif arch_name == "UnetPlusPlus":
            m = smp.UnetPlusPlus(encoder_name=enc, encoder_weights=None, in_channels=3, classes=1, activation=None)
        else:
            raise ValueError(f"Unknown arch: {arch_name}")

        missing, unexpected = m.load_state_dict(sd, strict=False)
        score = len(missing) + len(unexpected)
        print(f"[{arch_name} | {enc}] missing={len(missing)} unexpected={len(unexpected)} score={score}")

        if best is None or score < best["score"]:
            best = {
                "model": m,
                "encoder": enc,
                "score": score,
                "missing": missing,
                "unexpected": unexpected
            }

    if best["score"] > 150:
        raise RuntimeError(f"{arch_name} checkpoint load looks wrong. Score={best['score']}")

    return best["model"].to(device).eval(), best["encoder"], best

# Cell 9 — Load all 3 models

In [9]:
model_specs = {
    "model1_unet": {
        "arch": "Unet",
        "ckpt": MODEL1_CKPT,
    },
    "model2_deeplabv3plus": {
        "arch": "DeepLabV3Plus",
        "ckpt": MODEL2_CKPT,
    },
    "model3_unetplusplus": {
        "arch": "UnetPlusPlus",
        "ckpt": MODEL3_CKPT,
    },
}

loaded_models = {}

for name, spec in model_specs.items():
    print(f"\nLoading {name} ...")
    model, chosen_encoder, report = load_smp_model(spec["arch"], spec["ckpt"])
    loaded_models[name] = {
        "model": model,
        "arch": spec["arch"],
        "ckpt": spec["ckpt"],
        "encoder": chosen_encoder,
        "load_report": report,
    }

print("\n All models loaded")


Loading model1_unet ...
[Unet | efficientnet-b3] missing=494 unexpected=572 score=1066
[Unet | timm-efficientnet-b3] missing=0 unexpected=0 score=0

Loading model2_deeplabv3plus ...
[DeepLabV3Plus | efficientnet-b3] missing=0 unexpected=0 score=0
[DeepLabV3Plus | timm-efficientnet-b3] missing=494 unexpected=572 score=1066

Loading model3_unetplusplus ...
[UnetPlusPlus | efficientnet-b3] missing=0 unexpected=0 score=0
[UnetPlusPlus | timm-efficientnet-b3] missing=494 unexpected=572 score=1066

 All models loaded


# Cell 10 — Threshold sweep evaluator

In [10]:
def threshold_sweep_for_model(model_name, model, encoder_name):
    preprocess_fn = smp.encoders.get_preprocessing_fn(encoder_name, pretrained="imagenet")
    ds = CrackValDataset(val_df, img_size=IMG_SIZE, preprocess_fn=preprocess_fn)
    dl = DataLoader(ds, batch_size=4 if str(device) == "cpu" else 8, shuffle=False, num_workers=NUM_WORKERS)

    T = len(THRESHOLDS)
    thr_arr = np.array(THRESHOLDS, dtype=np.float32)

    sum_iou_all = np.zeros(T, dtype=np.float64)
    sum_iou_crack = np.zeros(T, dtype=np.float64)
    sum_dice_crack = np.zeros(T, dtype=np.float64)
    fp_no_crack = np.zeros(T, dtype=np.int64)

    n_total = 0
    n_crack = 0
    n_no_crack = 0

    for imgs, msks in dl:
        imgs = imgs.to(device)

        with torch.inference_mode():
            prob = torch.sigmoid(model(imgs)).cpu().numpy()[:, 0]  # (B,H,W)

        gt_batch = msks.numpy()[:, 0].astype(np.uint8)

        for b in range(prob.shape[0]):
            gt = gt_batch[b].astype(bool)
            pr_stack = prob[b][None, :, :] >= thr_arr[:, None, None]   # (T,H,W)

            gt_has = bool(gt.sum() > 0)
            pred_any = pr_stack.reshape(T, -1).any(axis=1)

            if gt_has:
                inter = np.logical_and(pr_stack, gt[None, :, :]).sum(axis=(1,2))
                union = np.logical_or(pr_stack, gt[None, :, :]).sum(axis=(1,2))
                denom = gt.sum() + pr_stack.sum(axis=(1,2))

                iou_arr = inter / np.maximum(union, 1)
                dice_arr = (2.0 * inter) / np.maximum(denom, 1)

                sum_iou_crack += iou_arr
                sum_dice_crack += dice_arr
                n_crack += 1
            else:
                iou_arr = (~pred_any).astype(np.float64)
                fp_no_crack += pred_any.astype(np.int64)
                n_no_crack += 1

            sum_iou_all += iou_arr
            n_total += 1

    out = pd.DataFrame({
        "model_name": model_name,
        "arch": loaded_models[model_name]["arch"],
        "encoder": encoder_name,
        "threshold": THRESHOLDS,
        "mean_iou_all": sum_iou_all / max(n_total, 1),
        "mean_iou_crack_only": sum_iou_crack / max(n_crack, 1),
        "mean_dice_crack_only": sum_dice_crack / max(n_crack, 1),
        "no_crack_fp_count": fp_no_crack,
        "no_crack_fp_rate": fp_no_crack / max(n_no_crack, 1),
        "n_total": n_total,
        "n_crack": n_crack,
        "n_no_crack": n_no_crack,
    })
    return out

# Cell 11 — Run sweep for all 3 models

In [11]:
all_rows = []

for model_name, info in loaded_models.items():
    print(f"\nRunning threshold sweep for {model_name} ...")
    df_model = threshold_sweep_for_model(
        model_name=model_name,
        model=info["model"],
        encoder_name=info["encoder"],
    )
    all_rows.append(df_model)

sweep_df = pd.concat(all_rows, ignore_index=True)
sweep_path = OUT_DIR / "phase8_threshold_sweep.csv"
sweep_df.to_csv(sweep_path, index=False)

print(" Saved:", sweep_path)
sweep_df.head()


Running threshold sweep for model1_unet ...


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]


Running threshold sweep for model2_deeplabv3plus ...


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]


Running threshold sweep for model3_unetplusplus ...
 Saved: /kaggle/working/output/phase8/phase8_threshold_sweep.csv


,model_name,arch,encoder,threshold,mean_iou_all,mean_iou_crack_only,mean_dice_crack_only,no_crack_fp_count,no_crack_fp_rate,n_total,n_crack,n_no_crack
0,model1_unet,Unet,timm-efficientnet-b3,0.1,0.578683,0.524623,0.671252,12,0.055046,1695,1477,218
1,model1_unet,Unet,timm-efficientnet-b3,0.2,0.591786,0.537628,0.682625,9,0.041284,1695,1477,218
2,model1_unet,Unet,timm-efficientnet-b3,0.3,0.597820,0.543876,0.688042,8,0.036697,1695,1477,218
3,model1_unet,Unet,timm-efficientnet-b3,0.4,0.601638,0.548257,0.691823,8,0.036697,1695,1477,218
4,model1_unet,Unet,timm-efficientnet-b3,0.5,0.604931,0.552036,0.695101,8,0.036697,1695,1477,218


# Cell 12 — Build threshold policy JSON

In [12]:
policy = {}

for model_name in sweep_df["model_name"].unique():
    sub = sweep_df[sweep_df["model_name"] == model_name].copy()

    best_iou_row = sub.sort_values(["mean_iou_crack_only", "no_crack_fp_rate"], ascending=[False, True]).iloc[0]
    best_dice_row = sub.sort_values(["mean_dice_crack_only", "no_crack_fp_rate"], ascending=[False, True]).iloc[0]

    iou_target = 0.98 * best_iou_row["mean_iou_crack_only"]
    risk_pool = sub[sub["mean_iou_crack_only"] >= iou_target].copy()
    risk_row = risk_pool.sort_values(["no_crack_fp_rate", "mean_iou_crack_only"], ascending=[True, False]).iloc[0]

    policy[model_name] = {
        "best_crack_iou_threshold": float(best_iou_row["threshold"]),
        "best_crack_iou_value": float(best_iou_row["mean_iou_crack_only"]),
        "best_crack_dice_threshold": float(best_dice_row["threshold"]),
        "best_crack_dice_value": float(best_dice_row["mean_dice_crack_only"]),
        "risk_controlled_threshold": float(risk_row["threshold"]),
        "risk_controlled_iou": float(risk_row["mean_iou_crack_only"]),
        "risk_controlled_fp_rate": float(risk_row["no_crack_fp_rate"]),
        "rule": "choose minimum no-crack FP rate among thresholds with crack-only IoU >= 98% of best crack-only IoU"
    }

policy_path = OUT_DIR / "phase8_threshold_policy.json"
policy_path.write_text(json.dumps(policy, indent=2), encoding="utf-8")

print(" Saved:", policy_path)
print(json.dumps(policy, indent=2))

 Saved: /kaggle/working/output/phase8/phase8_threshold_policy.json
{
  "model1_unet": {
    "best_crack_iou_threshold": 0.9,
    "best_crack_iou_value": 0.568641596904121,
    "best_crack_dice_threshold": 0.9,
    "best_crack_dice_value": 0.7104544583104387,
    "risk_controlled_threshold": 0.9,
    "risk_controlled_iou": 0.568641596904121,
    "risk_controlled_fp_rate": 0.01834862385321101,
    "rule": "choose minimum no-crack FP rate among thresholds with crack-only IoU >= 98% of best crack-only IoU"
  },
  "model2_deeplabv3plus": {
    "best_crack_iou_threshold": 0.4,
    "best_crack_iou_value": 0.5730325444459554,
    "best_crack_dice_threshold": 0.4,
    "best_crack_dice_value": 0.7142590736831291,
    "risk_controlled_threshold": 0.5,
    "risk_controlled_iou": 0.572807884519724,
    "risk_controlled_fp_rate": 0.0045871559633027525,
    "rule": "choose minimum no-crack FP rate among thresholds with crack-only IoU >= 98% of best crack-only IoU"
  },
  "model3_unetplusplus": {
    

# Cell 13 — Tradeoff plot (FP rate vs crack IoU)

In [13]:
plt.figure(figsize=(8,6))

for model_name in sweep_df["model_name"].unique():
    sub = sweep_df[sweep_df["model_name"] == model_name].sort_values("threshold")
    plt.plot(sub["no_crack_fp_rate"], sub["mean_iou_crack_only"], marker="o", label=model_name)

    # annotate thresholds
    for _, row in sub.iterrows():
        plt.text(row["no_crack_fp_rate"], row["mean_iou_crack_only"], f"{row['threshold']:.1f}", fontsize=8)

plt.xlabel("No-crack FP rate")
plt.ylabel("Mean IoU (crack-only)")
plt.title("Phase-8 Threshold Tradeoff: FP rate vs Crack IoU")
plt.legend()
plt.tight_layout()

tradeoff_path = OUT_DIR / "phase8_threshold_tradeoff_plot.png"
plt.savefig(tradeoff_path, dpi=220)
plt.close()

print(" Saved:", tradeoff_path)

 Saved: /kaggle/working/output/phase8/phase8_threshold_tradeoff_plot.png


# Cell 14 — Threshold curves plot

In [14]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for model_name in sweep_df["model_name"].unique():
    sub = sweep_df[sweep_df["model_name"] == model_name].sort_values("threshold")

    axes[0].plot(sub["threshold"], sub["mean_iou_crack_only"], marker="o", label=model_name)
    axes[1].plot(sub["threshold"], sub["no_crack_fp_rate"], marker="o", label=model_name)

axes[0].set_title("Threshold vs Crack-only IoU")
axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("Mean IoU (crack-only)")
axes[0].legend()

axes[1].set_title("Threshold vs No-crack FP Rate")
axes[1].set_xlabel("Threshold")
axes[1].set_ylabel("FP rate")
axes[1].legend()

plt.tight_layout()
curves_path = OUT_DIR / "phase8_threshold_curves.png"
plt.savefig(curves_path, dpi=220)
plt.close()

print(" Saved:", curves_path)

 Saved: /kaggle/working/output/phase8/phase8_threshold_curves.png


# Cell 15 — Quick readable summary table

In [15]:
summary_rows = []

for model_name, p in policy.items():
    summary_rows.append({
        "model_name": model_name,
        "best_iou_thr": p["best_crack_iou_threshold"],
        "best_iou": p["best_crack_iou_value"],
        "best_dice_thr": p["best_crack_dice_threshold"],
        "best_dice": p["best_crack_dice_value"],
        "risk_thr": p["risk_controlled_threshold"],
        "risk_iou": p["risk_controlled_iou"],
        "risk_fp_rate": p["risk_controlled_fp_rate"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,model_name,best_iou_thr,best_iou,best_dice_thr,best_dice,risk_thr,risk_iou,risk_fp_rate
0,model1_unet,0.9,0.568642,0.9,0.710454,0.9,0.568642,0.018349
1,model2_deeplabv3plus,0.4,0.573033,0.4,0.714259,0.5,0.572808,0.004587
2,model3_unetplusplus,0.4,0.580701,0.4,0.721607,0.5,0.580690,0.027523
